# Retrieval sweep on GPU

This notebook contains no pipeline logic. It clones the repo, installs
dependencies, and calls `run_sweep()`. Everything it runs also runs locally
with `python eval/run_sweep.py` — that is the point. Logic that lives only in
a notebook cannot be tested and cannot be reproduced after the session dies.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Internet → **On**.
Phone verification is required for both.

**Before the session ends:** run the last cell and download `sweep-artifacts.zip`.
`/kaggle/working` is deleted with the container.

The relevance gate stays off here. It is a service-layer control, and filtering
passages during a sweep would change what recall@5 measures.


In [ ]:
# 1. Get the code. Replace with your own repo URL.
REPO = "https://github.com/YOUR_USERNAME/docqa-arena.git"

import os, shutil
if os.path.exists("/kaggle/working/docqa-arena"):
    shutil.rmtree("/kaggle/working/docqa-arena")
!git clone -q $REPO /kaggle/working/docqa-arena
%cd /kaggle/working/docqa-arena
!ls

In [ ]:
# 2. Dependencies.
#
# Kaggle ships transformers 5.x with a matching huggingface_hub, and they work
# together. chromadb 0.5.23 pins tokenizers and huggingface_hub far lower; let
# pip resolve it normally and it downgrades both, which breaks
# sentence-transformers with `ImportError: cannot import name 'is_offline_mode'`.
# There is no single pin that satisfies both — the constraints genuinely conflict.
#
# So chromadb goes in with --no-deps and its real runtime requirements are added
# separately, leaving the ML stack untouched. onnxruntime is needed because
# chromadb instantiates its default ONNX embedding function at import time, even
# though this project always passes its own vectors.
!pip install -q --no-deps chromadb==0.5.23 2>&1 | tail -2
!pip install -q --no-deps posthog pypika chroma-hnswlib bcrypt overrides mmh3 2>&1 | tail -2
!pip install -q onnxruntime 2>&1 | tail -2
!pip install -q "opentelemetry-exporter-otlp-proto-grpc==1.27.0" 2>&1 | tail -3
!pip install -q openai==1.59.6 PyYAML==6.0.2 2>&1 | tail -2

# Verify the ML stack survived before spending GPU hours on it.
import transformers, huggingface_hub
print("transformers:", transformers.__version__, "hub:", huggingface_hub.__version__)
import chromadb
print("chromadb:", chromadb.__version__)
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
# 3. Secrets and paths.
#    Add LLM_API_KEY under Add-ons -> Secrets. Never paste a key into a cell —
#    Kaggle notebooks are public by default and cell output is saved with them.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["LLM_API_KEY"] = secrets.get_secret("LLM_API_KEY")
os.environ["JUDGE_API_KEY"] = os.environ["LLM_API_KEY"]
os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["JUDGE_BASE_URL"] = os.environ["LLM_BASE_URL"]

# Everything written at runtime goes here. Nothing else in the container survives.
os.environ["DOCQA_DATA_DIR"] = "/kaggle/working"
os.environ["ANONYMIZED_TELEMETRY"] = "False"
print("configured")


In [ ]:
# 4. Sanity check before spending GPU hours: does the golden set still resolve
#    against the corpus? If this fails, fix it now rather than after 18 configs.
!python scripts/resolve_spans.py --config configs/kaggle.yaml | tail -8

## A warning about judge quotas

The sweep makes roughly `31 questions x 18 configs x 2` LLM calls — one to
generate, one to judge — which is about 500k tokens. Free API tiers are smaller
than that: Groq's daily limit is 100k tokens for `llama-3.3-70b-versatile`.

**The sweep will run out of judge quota partway through.** That is expected and
not a failure. Retrieval metrics are unaffected, generated answers are all saved
in `evaluations.json`, and unjudged items are recorded as `null` with
`judge_coverage` in each summary rather than as a score of zero.

Finish the scoring locally afterwards, without re-running retrieval or
generation:

```
python scripts/rejudge.py --judge-model llama-3.3-70b-versatile
```

It deduplicates identical (question, answer, context) triples, caches verdicts to
disk, and resumes when a quota resets. Multiple keys can be rotated with
`--api-keys key1,key2`.


In [ ]:
# 5. Smoke-test one configuration first. If something is wrong with the API key
#    or the model download, find out in three minutes rather than three hours.
import sys; sys.path.insert(0, "/kaggle/working/docqa-arena")
from eval.run_sweep import run_sweep

run_sweep("configs/kaggle.yaml", only=["recursive__hybrid__norerank"])

In [ ]:
# 6. The full grid: 3 chunkers x 3 retrievers x 2 rerankers = 18 configurations.
#    Safe to re-run. Configs that already have a summary.json are skipped, so if
#    the session dies at config 11, restart the notebook and run this cell again.
summaries = run_sweep("configs/kaggle.yaml")
print(f"\n{len(summaries)} configurations complete")

In [ ]:
# 7. Leaderboard, read straight off disk.
!python scripts/leaderboard.py --config configs/kaggle.yaml --sort mrr

In [ ]:
# 8. Package everything for download. The results folder is the only thing that
#    survives this session; the GPU-built Chroma index is what lets the local
#    CPU-only container serve BGE-M3 embeddings without a GPU.
!python scripts/export_index.py --config configs/kaggle.yaml --out /kaggle/working/index.zip
!cd /kaggle/working && zip -qr sweep-artifacts.zip results index.zip && ls -lh sweep-artifacts.zip

print("\nDownload sweep-artifacts.zip from the Output panel on the right BEFORE closing this tab.")
print("Then, locally:")
print("  Expand-Archive sweep-artifacts.zip -DestinationPath kaggle-output")
print("  Copy-Item kaggle-output/results/* eval/results/ -Recurse -Force")
print("  python scripts/rejudge.py --judge-model llama-3.3-70b-versatile   # finish the judging")
print("  python scripts/leaderboard.py --sort mrr")
